In [ ]:
import os, warnings, sys
from pathlib import Path
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_preprocessing as du
import utils.model_utils as mu

importlib.reload(du)
importlib.reload(mu)

In [ ]:
SAVE_TICK_LEVEL_DATA = False

MODEL = "XGBoost"
MODEL_VERSION = "xgb_v1"
FEATURE_SET_ID = "microP_l1Imb_9Lags"
NORMALIZATION_ID = ""

XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    tree_method="hist",
    n_jobs=-1,
    random_state=0,
)

In [ ]:
# ============================================================
# Initialize Run
# ============================================================
warnings.filterwarnings("ignore")

PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"

# Per model family: model_outputs/XGBoost holds run_registry.csv plus
# DailyDiagnostics/<run_id> and TickLevel/<run_id>.
OUTPUT_ROOT = f"{PARENT}/model_outputs/XGBoost"

t = pd.read_parquet(f"{DATA_ROOT}/{du.SYMBOLS[0]}/{du.SAMPLE_DATES[0]}.parquet")

RUN_ID = mu.initialize_run(
    output_root=OUTPUT_ROOT,
    model_version=MODEL_VERSION,
    feature_set_id=FEATURE_SET_ID,
    normalization_id=NORMALIZATION_ID
)

DIRS = mu.create_output_dirs(
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
)

TICK_OUTPUT_DIR = DIRS["tick"]
DAILY_OUTPUT_DIR = DIRS["daily"]

daily_results = []

# Trading days used for the rolling walk-forward
SD = du.SAMPLE_DATES[:]

sample_cols = pd.read_parquet(f"{DATA_ROOT}/{du.SYMBOLS[0]}/{SD[0]}.parquet").columns

FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = list(sample_cols[sample_cols.str.startswith("T_")])

# ============================================================
# Main Loop
# ============================================================

GLOBAL_START = time.perf_counter()

# One symbol at a time: load all its days, then roll train(i) -> test(i+1).
for symbol in tqdm(du.SYMBOLS[:-1], desc="Processing symbols", unit="symbol"):
    day_cache = mu.load_day_cache(DATA_ROOT, symbol, SD, FEATURE_COLS, TARGET_COLS)

    # --------------------------------------------------------
    # Rolling train/test
    # --------------------------------------------------------

    for i in tqdm(range(len(SD) - 1), desc=f"{symbol}", leave=False, unit="split"):

        train_day = SD[i]
        test_day = SD[i+1]

        train_cache = day_cache[train_day]
        test_cache = day_cache[test_day]

        X_train = train_cache["X"]
        X_test = test_cache["X"]

        Y_train = train_cache["Y"]
        Y_test = test_cache["Y"]

        # Multi-output: XGBoost fits one booster per target internally
        # when Y_train is 2D.
        model = XGBRegressor(**XGB_PARAMS)
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_test).astype(np.float32)

        # Residuals
        RESID = (Y_test - Y_pred).astype(np.float32)

        if SAVE_TICK_LEVEL_DATA:
            mu.save_tick_residuals(
                resid=RESID,
                timestamps=test_cache["timestamp"],
                target_cols=TARGET_COLS,
                tick_output_dir=TICK_OUTPUT_DIR,
                test_day=test_day,
                symbol=symbol
            )

        daily_results += mu.daily_diagnostic_rows(
            resid=RESID,
            Y_test=Y_test,
            target_cols=TARGET_COLS,
            train_day=train_day,
            test_day=test_day,
            symbol=symbol,
            run_id=RUN_ID,
            n_train=X_train.shape[0],
            n_test=X_test.shape[0]
        )

tqdm.write(f"\nTOTAL PIPELINE TIME: {time.perf_counter()-GLOBAL_START:.2f}s")
# ============================================================
# Save Outputs
# ============================================================

mu.save_table(
    df=pd.DataFrame(daily_results),
    root_dir=DAILY_OUTPUT_DIR,
    filename="daily_diagnostics.parquet",
)

In [ ]:
daily_out = pd.read_parquet(f"{DAILY_OUTPUT_DIR}/daily_diagnostics.parquet")
daily_out[daily_out["mse_ratio"] < 1]